In [2]:
pip install kagglehub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [kagglehub]
Note: you may need to restart the kernel to use updated packages.


In [1]:
# Loading the dataset in - since it is so large, it must be done using batch processing

import pandas as pd
import kagglehub


csv_path = kagglehub.dataset_download(
    "sobhanmoosavi/us-accidents",
    path="US_Accidents_March23.csv",
)

batch_size = 10000
batches = []
batch_num = 0

for batch in pd.read_csv(csv_path, chunksize=batch_size, low_memory=False):
    batches.append(batch)
    batch_num += 1
    if batch_num % 20 == 0:
        print(f"Processed {batch_num * batch_size} rows")

df = pd.concat(batches, ignore_index=True)
print(df.shape)

100%|██████████| 653M/653M [00:36<00:00, 18.9MB/s] 

Extracting zip of US_Accidents_March23.csv...


Processed 200000 rows
Processed 400000 rows
Processed 600000 rows
Processed 800000 rows
Processed 1000000 rows
Processed 1200000 rows
Processed 1400000 rows
Processed 1600000 rows
Processed 1800000 rows
Processed 2000000 rows
Processed 2200000 rows
Processed 2400000 rows
Processed 2600000 rows
Processed 2800000 rows
Processed 3000000 rows
Processed 3200000 rows
Processed 3400000 rows
Processed 3600000 rows
Processed 3800000 rows
Processed 4000000 rows
Processed 4200000 rows
Processed 4400000 rows
Processed 4600000 rows
Processed 4800000 rows
Processed 5000000 rows
Processed 5200000 rows
Processed 5400000 rows
Processed 5600000 rows
Processed 5800000 rows
Processed 6000000 rows
Processed 6200000 rows
Processed 6400000 rows
Processed 6600000 rows
Processed 6800000 rows
Processed 7000000 rows
Processed 7200000 rows
Processed 7400000 rows
Processed 7600000 rows
(7728394, 46)


In [2]:
# Finding the top 10 cities with the most accidents
df["City"].value_counts().head(10)

City
Miami          186917
Houston        169609
Los Angeles    156491
Charlotte      138652
Dallas         130939
Orlando        109733
Austin          97359
Raleigh         86079
Nashville       72930
Baton Rouge     71588
Name: count, dtype: int64

In [3]:
# To trim the dataset, we choose the top 3 cities with the most accidents: Miami, Houston, and Los Angeles
df_trimmed = df[df["City"].isin(["Miami", "Houston", "Los Angeles"])]
df_trimmed

,ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),...,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
42866,A-42867,Source2,2,2016-06-21 10:46:30,2016-06-21 11:27:00,34.078926,-118.289040,NaN,NaN,0.000,...,False,False,False,False,False,False,Day,Day,Day,Day
42867,A-42868,Source2,3,2016-06-21 10:49:21,2016-06-21 11:34:21,34.091179,-118.239471,NaN,NaN,0.000,...,False,False,False,False,False,False,Day,Day,Day,Day
42881,A-42882,Source2,3,2016-06-21 10:51:45,2016-06-21 11:36:45,34.037239,-118.309074,NaN,NaN,0.000,...,False,True,False,False,False,False,Day,Day,Day,Day
42883,A-42884,Source2,3,2016-06-21 10:56:24,2016-06-21 11:34:00,34.027458,-118.274490,NaN,NaN,0.000,...,False,False,False,False,False,False,Day,Day,Day,Day
42898,A-42899,Source2,3,2016-06-21 11:30:46,2016-06-21 12:00:46,33.947544,-118.279434,NaN,NaN,0.000,...,False,False,False,False,False,False,Day,Day,Day,Day
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7728130,A-7777498,Source1,3,2019-08-23 17:32:09,2019-08-23 18:02:03,29.971810,-95.562340,29.96023,-95.546779,1.228,...,False,False,False,False,False,False,Day,Day,Day,Day
7728357,A-7777725,Source1,3,2019-08-23 04:04:48,2019-08-23 04:33:53,34.075790,-118.276680,34.07431,-118.272250,0.273,...,False,False,False,False,False,False,Night,Night,Night,Night
7728361,A-7777729,Source1,2,2019-08-23 12:52:31,2019-08-23 13:20:14,34.023790,-118.276390,34.02576,-118.275290,0.150,...,False,False,False,False,False,False,Day,Day,Day,Day
7728364,A-7777732,Source1,2,2019-08-23 13:42:50,2019-08-23 14:10:06,34.070610,-118.263910,34.06974,-118.261550,0.148,...,False,False,False,False,False,False,Day,Day,Day,Day


In [12]:
# select columns to be dropped
columns_dropped = [
    # zero/near-zero variance
    'Turning_Loop',
    'Roundabout',
    'No_Exit',
    'Traffic_Calming',
    'Bump',

    # redundant or irrelevant
    'Country',
    'State',
    'End_Lat',
    'End_Lng',
    'Airport_Code',
    'Weather_Timestamp',
    'County',
    'Timezone',
    'Civil_Twilight',
    'Nautical_Twilight',
    'Astronomical_Twilight',
    'Wind_Chill(F)',
    'ID',
    'Description',
    'Street',
]

pd.set_option('display.max_info_columns', 30)

# # create a copy to keep a primary df and apply transformation
d2 = df_trimmed.copy().drop(columns_dropped, axis=1)

print(f"Columns before: {df_trimmed.shape[1]}")
print(f"Columns after: {d2.shape[1]}")
print(f"Remaining columns: {list(d2.columns)}")

Columns before: 46
Columns after: 26
Remaining columns: ['Source', 'Severity', 'Start_Time', 'End_Time', 'Start_Lat', 'Start_Lng', 'Distance(mi)', 'City', 'Zipcode', 'Temperature(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Direction', 'Wind_Speed(mph)', 'Precipitation(in)', 'Weather_Condition', 'Amenity', 'Crossing', 'Give_Way', 'Junction', 'Railway', 'Station', 'Stop', 'Traffic_Signal', 'Sunrise_Sunset']


In [13]:
# checking for null values
d2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 513017 entries, 42866 to 7728376
Data columns (total 26 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Source             513017 non-null  object 
 1   Severity           513017 non-null  int64  
 2   Start_Time         513017 non-null  object 
 3   End_Time           513017 non-null  object 
 4   Start_Lat          513017 non-null  float64
 5   Start_Lng          513017 non-null  float64
 6   Distance(mi)       513017 non-null  float64
 7   City               513017 non-null  object 
 8   Zipcode            513017 non-null  object 
 9   Temperature(F)     507387 non-null  float64
 10  Humidity(%)        507117 non-null  float64
 11  Pressure(in)       509236 non-null  float64
 12  Visibility(mi)     508298 non-null  float64
 13  Wind_Direction     506361 non-null  object 
 14  Wind_Speed(mph)    471323 non-null  float64
 15  Precipitation(in)  371056 non-null  float64
 16  We

In [10]:
d2 = d2.dropna(axis=0)
d2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 363932 entries, 56613 to 7728376
Data columns (total 25 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Source             363932 non-null  object 
 1   Severity           363932 non-null  int64  
 2   Start_Time         363932 non-null  object 
 3   End_Time           363932 non-null  object 
 4   Start_Lat          363932 non-null  float64
 5   Start_Lng          363932 non-null  float64
 6   Distance(mi)       363932 non-null  float64
 7   City               363932 non-null  object 
 8   Temperature(F)     363932 non-null  float64
 9   Humidity(%)        363932 non-null  float64
 10  Pressure(in)       363932 non-null  float64
 11  Visibility(mi)     363932 non-null  float64
 12  Wind_Direction     363932 non-null  object 
 13  Wind_Speed(mph)    363932 non-null  float64
 14  Precipitation(in)  363932 non-null  float64
 15  Weather_Condition  363932 non-null  object 
 16  Am

In [15]:
d2.isna().sum()

Source                    0
Severity                  0
Start_Time                0
End_Time                  0
Start_Lat                 0
Start_Lng                 0
Distance(mi)              0
City                      0
Zipcode                   0
Temperature(F)         5630
Humidity(%)            5900
Pressure(in)           3781
Visibility(mi)         4719
Wind_Direction         6656
Wind_Speed(mph)       41694
Precipitation(in)    141961
Weather_Condition      4583
Amenity                   0
Crossing                  0
Give_Way                  0
Junction                  0
Railway                   0
Station                   0
Stop                      0
Traffic_Signal            0
Sunrise_Sunset            6
dtype: int64